### For each language (ASL and English), pool individual results across Studies 1 & 2 and calculate group-level T-statistic
### Correct 500-parcel results with Sign Flip Permutation test, return regions where permuted p < 0.05

In [1]:
# environment: fmri_stats.yml

import numpy as np
import pandas as pd
import pickle
from copy import deepcopy

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap


base_dir = '/dartfs-hpc/rc/lab/K/KraemerD/ASL1-2_combined/2026/OSF Fig Share/' #main repo folder
rsa_dir = base_dir+'data/rsa/'
out_dir = base_dir+'figures/'

%run '/dartfs-hpc/rc/lab/K/KraemerD/ASL1-2_combined/2026/scripts/asl_combo_helpers.py'

In [4]:
# stat params
pval=0.05
res = 'p_perm_maxT' # which result should we use? (must be a key in the signfliptest results dict)
sigparcs = 'signflip' # how should this result be labelled in the file names?

# model
model = 'word2vec'
dm_type = '_zscored_weighted'
plot='TRUE'

# ASL

In [6]:
# Create a master table of correlation values from already-computed RSAs:
rsa_dict = {}
rsa_dict[model] = [[]]*500

files=[
    # rsa_dir+'corrs/asl1_'+model+dm_type+'_grpASL_asl_Schaefer500_corrs_normed.pkl',
      rsa_dir+'corrs/asl2_'+model+dm_type+'_grp1_asl_words2_Schaefer500_corrs_normed.pkl',
      rsa_dir+'corrs/asl2_'+model+dm_type+'_grp2_asl_words1_Schaefer500_corrs_normed.pkl']

for s in range(len(files)):
    corrs = pickle.load(open(files[s], 'rb' ))

    for parc in range(500):
        rsa_dict[model][parc] = rsa_dict[model][parc]+list(corrs[parc])

df = pickle.load(open(files[0], 'rb' ))

if len(files) > 1:
    for f in range(1,len(files)):
        corrs = pickle.load(open(files[0], 'rb' ))
        df = pd.concat([df, corrs])

results = one_sample_perm_test_signflip(df, alternative='two-sided')
print("lowest corrected p:",np.min(results['p_perm_maxT']))

parc_vals = [0]*500
asl_parcs = []
count = 0

ts =[]

for parc in range(len(parc_vals)):
    if results[res][parc]<pval and results['t_obs'][parc]>0:
        parc_vals[parc] = results['t_obs'][parc]
        ts.append(results['t_obs'][parc])
        asl_parcs.append(parc)
        count+=1

print(count,"sig. parcels")

# make & save heatmap
# this snippet creates the heatmap presented in Figure 2A and Supp. Fig. 1
if plot == 'TRUE' and count > 0:
    rh_masked, lh_masked = parc_list_to_surf(parc_vals, 500)
    
    fn = out_dir+'ASL-combo_ASL_'+model+dm_type+'_'+sigparcs+'_parcels_p'+str(pval)
    four_panel_surfplot(rh_masked, lh_masked,fn,title="ASL-Combo-Unknown_ASL_"+model+'_'+sigparcs+"_p<"+str(pval),
                        colormap='coolwarm',bg_on_data=True,cmap_method='center')


lowest corrected p: 9.999000099990002e-05
1 sig. parcels
2000 1700


In [9]:
print("Selected ASL RSA Parcels") # parcel INDICES (0-499, NOT true 1-500 labels)
print(asl_parcs)

Selected ASL RSA Parcels
[0, 1, 5, 26, 28, 31, 35, 50, 53, 89, 91, 106, 107, 147, 154, 155, 174, 198, 199, 200, 201, 202, 204, 238, 244, 246, 250, 259, 270, 273, 276, 277, 281, 283, 291, 301, 308, 335, 338, 345, 356, 369, 380, 382, 383, 388, 414, 436, 441, 452, 455, 456, 457, 490]


# Eng

In [3]:
# Create a master table of correlation values from already-computed RSAs:
rsa_dict = {}
rsa_dict[model] = [[]]*500

files=[rsa_dir+'corrs/asl1_'+model+dm_type+'_grpASL_eng_Schaefer500_corrs_normed.pkl',
      rsa_dir+'corrs/asl2_'+model+dm_type+'_allsubs_eng_words1_Schaefer500_corrs_normed.pkl',
      rsa_dir+'corrs/asl2_'+model+dm_type+'_allsubs_eng_words2_Schaefer500_corrs_normed.pkl',]

for s in range(len(files)):
    corrs = pickle.load(open(files[s], 'rb' ))

    for parc in range(500):
        rsa_dict[model][parc] = rsa_dict[model][parc]+list(corrs[parc])

df = pickle.load(open(files[0], 'rb' ))

if len(files) > 1:
    for f in range(1,len(files)):
        corrs = pickle.load(open(files[0], 'rb' ))
        df = pd.concat([df, corrs])

results = one_sample_perm_test_signflip(df, alternative='greater')
print("lowest corrected p:",np.min(results['p_perm_maxT']))

parc_vals = [0]*500
eng_parcs = []
count = 0

for parc in range(len(parc_vals)):
    if results[res][parc]<pval and results['t_obs'][parc]>0:
        parc_vals[parc] = results['t_obs'][parc]
        eng_parcs.append(parc)
        count+=1

print(count,"parcels")

# This heatmap is presented in Supp. Fig 1
# make & save heatmap
if plot == 'TRUE' and count > 0:
    rh_masked, lh_masked = parc_list_to_surf(parc_vals, 500)
    
    fn = out_dir+'ASL-combo_ENG_'+model+dm_type+'_'+sigparcs+'_parcels_p'+str(pval)
    four_panel_surfplot(rh_masked, lh_masked,fn,title="ASL-Combo_ENG_"+model+'_'+sigparcs+"_p<"+str(pval),
                        colormap='coolwarm',bg_on_data=True,cmap_method='center')


lowest corrected p: 9.999000099990002e-05
26 parcels


In [4]:
parc_vals = np.zeros(500)
parc_vals[200] = 1
parc_vals[306]=2

colors = ['#17becf','#e377c2']
customcmap = ListedColormap(colors)

In [5]:
rh_masked, lh_masked = parc_list_to_surf(parc_vals, 500)

fn = out_dir+'example_parcels'
four_panel_surfplot(rh_masked, lh_masked,fn,title="example parc",
                    colormap=customcmap,bg_on_data=True,cmap_method='range')

2000 1700


'Saved figure to /dartfs-hpc/rc/lab/K/KraemerD/ASL1-2_combined/2026/OSF Fig Share/figures/cropped/example_parcels_cropped.png'

In [11]:
print("Selected ENG RSA Parcels") # parcel INDICES (0-499, NOT true 1-500 labels)
print(eng_parcs)

Selected ENG RSA Parcels
[6, 23, 24, 84, 95, 97, 100, 102, 116, 127, 178, 181, 198, 228, 236, 252, 257, 264, 279, 311, 317, 352, 373, 375, 386, 446]
